In [ ]:
import os 
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
from typing import List
import time
import os
import numpy as np
import time
import os
import numpy as np
import torch
import pickle
import argparse
from uuid import uuid4

from torch.utils.data import DataLoader

import sys
sys.path.append('..')
sys.path.append('../..')
sys.path.append('../../..')
from reactot.trainer.pl_trainer import SBModule
from reactot.dataset.transition1x import ProcessedTS1x
from reactot.analyze.rmsd import batch_rmsd
from reactot.analyze.geomopt import calc_deltaE, compute_efh
from reactot.evaluate.utils import (
    set_new_schedule,
    inplaint_batch,
    batch_ts_deltaE,
)
from reactot.utils.sampling_tools import write_tmp_xyz


In [ ]:
from Utils import rmsd_loss, calculate_efh, AU2EV
import yaml
from easydict import EasyDict

from ase import Atoms

In [ ]:
parser = argparse.ArgumentParser(description='Training Transition1x dynamics')
parser.add_argument('--config_file_flow', required=True)
parser.add_argument('--log_prefix', default='logs')
parser.add_argument('--notes', default=' ')
parser.add_argument('--device', default='cuda')
parser.add_argument('--resume_status', default=' ')
parser.add_argument('--flow', default=' ')
args = parser.parse_args(['--config_file_flow', "../../Configs/Dynamics_rgd1.yml",
                          '--device', 'cuda',
                          '--flow', '']) # trained TS-DFM checkpoint

In [ ]:
device = args.device

In [ ]:
def load_model(
    checkpoint_path,
    stage
):
    print (checkpoint_path)
    model = SBModule.load_from_checkpoint(
        checkpoint_path=checkpoint_path,
        map_location=args.device,
    )
    model = model.eval()
    model = model.to(args.device)

    model.training_config["use_sampler"] = False
    model.training_config["swapping_react_prod"] = False
    model.training_config["datadir"] = "./data/RGD1"

    model.setup(stage, device=args.device, swapping_react_prod=False)
    return model

opt = {
    "batch_size": 1,
    "nfe": 10,
    "solver": "ode",
    "checkpoint": "", # react-ot checkpoint
    "order": 1,
    "diz": "linear",
    "method": "midpoint",
    "atol": 1e-2,
    "rtol": 1e-2
}
opt = EasyDict(opt)

In [ ]:
import torch
from torch import nn, optim
import argparse
import sys
sys.path.append('./')
sys.path.append('../')
import os
import yaml
from easydict import EasyDict
from collections import OrderedDict
import random
import numpy as np
import pickle as pkl
import h5py

from torch.utils.data import IterableDataset

from Data.RGD1 import generate_dataloader_dynamics
from Model.model import DistFlowMatchingNetwork, ODEWrapper2, InterpNetwork
from Model.backbone import generate_backbone
from Model.head import generate_head
from Model.model import MDNet
from Utils import get_logger, get_new_log_dir, seed_all, Kabsch_alignment, rmsd_loss, generate_fully_connected, create_angular_index, calculate_angle, d_mae_loss, calculate_efh, AU2EV, pairwise_dist_to_coord
import gc

from torch_scatter import scatter_mean, scatter_add
from torch.optim import LBFGS

from torch_geometric.data import Data, DataLoader

import pickle

from ase import Atoms

In [ ]:
def calculate_loss_coord(adj_matrix, coord, edge_index):
    src = edge_index[0]
    dst = edge_index[1]
    diff = coord[src] - coord[dst]
    dists = torch.norm(diff, p=2, dim=-1)
    loss = torch.sum(torch.square(dists - adj_matrix) * torch.pow(1 / (adj_matrix + 1e-6), 2))
    return loss

In [ ]:
dtype = torch.float32

config_path=args.config_file_flow
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)
config = EasyDict(config)
config.notes = args.notes

device = args.device

seed_all(config.train.seed)
torch.backends.cudnn.benchmark = True

config.data.batch_size = 1

In [ ]:
dynamic_model = DistFlowMatchingNetwork(**config.dynamic_model.parameters)
dynamic_model.load_state_dict(torch.load(args.flow)['model'])

dynamic_model = dynamic_model.to(device)
ode = ODEWrapper2(dynamic_model)

dynamic_model.eval()

In [ ]:
def pathpred(x, reactant_pos, product_pos, transition_state_pos):
    timestep = 0.1

    x = x.to(device)
    reactant_pos = reactant_pos.to(device)
    product_pos = product_pos.to(device)

    transition_state_pos = transition_state_pos.to(device)

    batch = torch.zeros(x.shape[0], dtype=torch.int64, device=device)

    src, dst = generate_fully_connected(batch)

    edge_index = torch.concat([src.unsqueeze(0), dst.unsqueeze(0)], dim=0)

    dist_reactant = torch.norm(reactant_pos[src] - reactant_pos[dst], p=2, dim=-1)
    dist_product = torch.norm(product_pos[src] - product_pos[dst], p=2, dim=-1)

    interp_t = torch.ones_like(batch, dtype=torch.float32) * 0.5

    dist_lin_interp = (1 - interp_t[batch[src]]) * torch.norm(reactant_pos[src] - reactant_pos[dst], p=2, dim=-1) + interp_t[batch[src]] * torch.norm(product_pos[src] - product_pos[dst], p=2, dim=-1)

    dist_transition_state = torch.norm(transition_state_pos[src] - transition_state_pos[dst], p=2, dim=-1)

    dist_transition_state_pred = ode(x, edge_index, dist_reactant, dist_product, dist_lin_interp, batch)

    curr_dist_matrix = dist_transition_state_pred.detach()
    
    pred_pos = torch.tensor((reactant_pos + product_pos) / 2, dtype=curr_dist_matrix.dtype, requires_grad=True)
    pred_pos.requires_grad = True
    optimizer = LBFGS([pred_pos], max_iter=100)
    def closure():
        optimizer.zero_grad()
        loss = calculate_loss_coord(curr_dist_matrix, pred_pos, edge_index)
        loss.backward()
        return loss
    optimizer.step(closure)

    loss_rmsd = rmsd_loss(pred_pos, transition_state_pos, torch.zeros(pred_pos.shape[0], dtype=torch.int64, device=device))
    loss_dmae = d_mae_loss(pred_pos, transition_state_pos, torch.zeros(pred_pos.shape[0], dtype=torch.int64, device=device))

    return loss_rmsd.cpu(), loss_dmae.cpu(), pred_pos.cpu()

In [ ]:
model = load_model(opt.checkpoint, 'test')

test_loader = model.test_dataloader(bz=opt.batch_size)
model.nfe = opt.nfe
model.ddpm.opt = opt # hack :)

rmsds_list, dmaes_list, deltaEs_list = [], [], []
deltaEs_dft_list = []
reactant_product_pos = []
transition_state_pos_true = []
transition_state_pos_pred = []

atom_types = []
true_energy_barrier = []
res_dft = []

rmsds_list2, dmaes_list2 = [], []
transition_state_pos_pred2 = []
for ii, batch in enumerate(test_loader):
    pred_transition_state_pos, reps, conds, ediff, rmsds, dmaes = model.eval_sample_batch(
    batch,
    test=True,
    )  # 30s for nfe=100

    transition_state_pos_pred.append(pred_transition_state_pos)
    
    true_energy_barrier.append(ediff)
    
    atom_type = reps[0]['charge'].type(torch.long).squeeze().to(device)
    reactant_pos = reps[0]['pos'].to(device)
    product_pos = reps[2]['pos']
    trans_pos = reps[1]['pos']
    transition_state_pos_true.append(trans_pos)

    rmsd_2, dmae_2, pred_transition_state_pos_2 = pathpred(atom_type, reactant_pos, product_pos, trans_pos)

    rmsds_list2.append(rmsd_2.detach().cpu().numpy().item())
    dmaes_list2.append(dmae_2.detach().cpu().numpy().item())
    transition_state_pos_pred2.append(pred_transition_state_pos_2.detach().cpu().numpy())

    reactant_product_pos.append((reactant_pos.cpu().numpy(), product_pos.cpu().numpy()))
    atom_types.append(atom_type.cpu().numpy())

    batch_num = torch.zeros_like(atom_type)
    pred_transition_state_pos = pred_transition_state_pos.to(device)

    atom_reactant = Atoms(numbers=atom_type.cpu().numpy(), positions=reactant_pos.detach().cpu().numpy())
    atom_trans = Atoms(numbers=atom_type.cpu().numpy(), positions=pred_transition_state_pos.detach().cpu().numpy())


    rmsds = rmsd_loss(pred_transition_state_pos, trans_pos, batch_num).cpu().numpy()

    rmsds_list.append(rmsds.item())
    dmaes_list.append(dmaes[0])

print(np.mean(rmsds_list), np.median(rmsds_list))
print(np.mean(dmaes_list), np.median(dmaes_list))

In [ ]:
res_id = {
    'pred_transition_state_pos': transition_state_pos_pred,
    'true_transition_state_pos': transition_state_pos_true,
    'dft_res': res_dft,
    'reactant_product_pos': reactant_product_pos,
    'atom_types': atom_types,
    'true_energy_barrier': true_energy_barrier,
    'rmsds': rmsds_list,
    'dmaes': dmaes_list,
    'rmsds2': rmsds_list2,
    'dmaes2': dmaes_list2,
    'pred_transition_state_pos2': transition_state_pos_pred2,
}

In [ ]:
model = load_model(opt.checkpoint, 'test_ood_type')

test_loader = model.test_dataloader(bz=opt.batch_size)
model.nfe = opt.nfe
model.ddpm.opt = opt # hack :)

rmsds_list, dmaes_list, deltaEs_list = [], [], []
deltaEs_dft_list = []
reactant_product_pos = []
transition_state_pos_true = []
transition_state_pos_pred = []
atom_types = []
true_energy_barrier = []
res_dft = []

rmsds_list2, dmaes_list2 = [], []
transition_state_pos_pred2 = []
for ii, batch in enumerate(test_loader):
    pred_transition_state_pos, reps, conds, ediff, rmsds, dmaes = model.eval_sample_batch(
    batch,
    test=True,
    )  # 30s for nfe=100

    transition_state_pos_pred.append(pred_transition_state_pos)
    
    true_energy_barrier.append(ediff)
    
    atom_type = reps[0]['charge'].type(torch.long).squeeze().to(device)
    reactant_pos = reps[0]['pos'].to(device)
    product_pos = reps[2]['pos']
    trans_pos = reps[1]['pos']
    transition_state_pos_true.append(trans_pos)

    rmsd_2, dmae_2, pred_transition_state_pos_2 = pathpred(atom_type, reactant_pos, product_pos, trans_pos)

    rmsds_list2.append(rmsd_2.detach().cpu().numpy().item())
    dmaes_list2.append(dmae_2.detach().cpu().numpy().item())
    transition_state_pos_pred2.append(pred_transition_state_pos_2.detach().cpu().numpy())

    reactant_product_pos.append((reactant_pos.cpu().numpy(), product_pos.cpu().numpy()))
    atom_types.append(atom_type.cpu().numpy())

    batch_num = torch.zeros_like(atom_type)
    pred_transition_state_pos = pred_transition_state_pos.to(device)

    atom_reactant = Atoms(numbers=atom_type.cpu().numpy(), positions=reactant_pos.detach().cpu().numpy())
    atom_trans = Atoms(numbers=atom_type.cpu().numpy(), positions=pred_transition_state_pos.detach().cpu().numpy())


    rmsds = rmsd_loss(pred_transition_state_pos, trans_pos, batch_num).cpu().numpy()

    rmsds_list.append(rmsds.item())
    dmaes_list.append(dmaes[0])

print(np.mean(rmsds_list), np.median(rmsds_list))
print(np.mean(dmaes_list), np.median(dmaes_list))

In [ ]:
res_ood_type = {
    'pred_transition_state_pos': transition_state_pos_pred,
    'true_transition_state_pos': transition_state_pos_true,
    'dft_res': res_dft,
    'reactant_product_pos': reactant_product_pos,
    'atom_types': atom_types,
    'true_energy_barrier': true_energy_barrier,
    'rmsds': rmsds_list,
    'dmaes': dmaes_list,
    'rmsds2': rmsds_list2,
    'dmaes2': dmaes_list2,
    'pred_transition_state_pos2': transition_state_pos_pred2,
}

In [ ]:
model = load_model(opt.checkpoint, 'test_ood_size')

test_loader = model.test_dataloader(bz=opt.batch_size)
model.nfe = opt.nfe
model.ddpm.opt = opt # hack :)

rmsds_list, dmaes_list, deltaEs_list = [], [], []
deltaEs_dft_list = []
reactant_product_pos = []
transition_state_pos_true = []
transition_state_pos_pred = []
atom_types = []
true_energy_barrier = []
res_dft = []

rmsds_list2, dmaes_list2 = [], []
transition_state_pos_pred2 = []
for ii, batch in enumerate(test_loader):
    pred_transition_state_pos, reps, conds, ediff, rmsds, dmaes = model.eval_sample_batch(
    batch,
    test=True,
    )  # 30s for nfe=100

    transition_state_pos_pred.append(pred_transition_state_pos)
    
    true_energy_barrier.append(ediff)
    
    atom_type = reps[0]['charge'].type(torch.long).squeeze().to(device)
    reactant_pos = reps[0]['pos'].to(device)
    product_pos = reps[2]['pos']
    trans_pos = reps[1]['pos']
    transition_state_pos_true.append(trans_pos)

    rmsd_2, dmae_2, pred_transition_state_pos_2 = pathpred(atom_type, reactant_pos, product_pos, trans_pos)

    rmsds_list2.append(rmsd_2.detach().cpu().numpy().item())
    dmaes_list2.append(dmae_2.detach().cpu().numpy().item())
    transition_state_pos_pred2.append(pred_transition_state_pos_2.detach().cpu().numpy())

    reactant_product_pos.append((reactant_pos.cpu().numpy(), product_pos.cpu().numpy()))
    atom_types.append(atom_type.cpu().numpy())

    batch_num = torch.zeros_like(atom_type)
    pred_transition_state_pos = pred_transition_state_pos.to(device)

    atom_reactant = Atoms(numbers=atom_type.cpu().numpy(), positions=reactant_pos.detach().cpu().numpy())
    atom_trans = Atoms(numbers=atom_type.cpu().numpy(), positions=pred_transition_state_pos.detach().cpu().numpy())

    rmsds = rmsd_loss(pred_transition_state_pos, trans_pos, batch_num).cpu().numpy()

    rmsds_list.append(rmsds.item())
    dmaes_list.append(dmaes[0])

print(np.mean(rmsds_list), np.median(rmsds_list))
print(np.mean(dmaes_list), np.median(dmaes_list))

In [ ]:
res_ood_size = {
    'pred_transition_state_pos': transition_state_pos_pred,
    'true_transition_state_pos': transition_state_pos_true,
    'dft_res': res_dft,
    'reactant_product_pos': reactant_product_pos,
    'atom_types': atom_types,
    'true_energy_barrier': true_energy_barrier,
    'rmsds': rmsds_list,
    'dmaes': dmaes_list,
    'rmsds2': rmsds_list2,
    'dmaes2': dmaes_list2,
    'pred_transition_state_pos2': transition_state_pos_pred2,
}

In [ ]:
res = {
    'id_test': res_id,
    'ood_test_type': res_ood_type,
    'ood_test_size': res_ood_size
}

import pickle
pickle.dump(res, open('rgd1_test_results.pkl', 'wb'))